In [ ]:
!pip install -q datasets

from datasets import load_dataset

dataset = load_dataset("sepidmnorozy/Turkish_sentiment")
print(dataset)
print(dataset["train"][0])

In [ ]:
from collections import Counter

labels = dataset["train"]["label"]
counts = Counter(labels)
for label, count in sorted(counts.items()):
    sentiment = "Positive" if label == 1 else "Negative"
    print(f"{sentiment}: {count} ({count/len(labels)*100:.1f}%)")

# Look at a few examples
for i in range(5):
    ex = dataset["train"][i]
    sentiment = "Positive" if ex["label"] == 1 else "Negative"
    print(f"\n{sentiment}: {ex['text']}")

In [ ]:
!pip install -q transformers

from transformers import pipeline
import numpy as np

# English sentiment model, never seen Turkish
english_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Test on Turkish examples
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

correct = 0
for text, true_label in zip(test_texts, test_labels):
    result = english_model(text, truncation=True)[0]
    pred = 1 if result["label"] == "POSITIVE" else 0
    if pred == true_label:
        correct += 1

zero_shot_accuracy = correct / len(test_labels)
print(f"Zero-shot accuracy (English model on Turkish): {zero_shot_accuracy:.4f}")
print(f"Random baseline would be: 0.5000")

In [ ]:
!pip install -q evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=128)

tokenized = dataset.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"]
    }

args = TrainingArguments(
    output_dir="turkish-sentiment",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate(tokenized["test"])
print(f"mBERT Fine-tuned Accuracy: {results['eval_accuracy']:.4f}")
print(f"mBERT Fine-tuned F1:       {results['eval_f1']:.4f}")
print()
print("=== CROSS-LINGUAL TRANSFER SUMMARY ===")
print(f"English model zero-shot on Turkish: 0.4171  (worse than random)")
print(f"mBERT fine-tuned on Turkish:        {results['eval_accuracy']:.4f}")
print(f"Recovery gap:                       +{results['eval_accuracy']-0.4171:.4f}")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
trainer.push_to_hub("turkish-sentiment-mbert")

In [ ]:
!pip install -q gradio

import gradio as gr
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

def predict(text):
    result = classifier(text, truncation=True)[0]
    label = "Positive 😊" if result["label"] == "POSITIVE" else "Negative 😞"
    return f"{label} — {result['score']*100:.1f}% confidence"

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(
        label="Turkish Review",
        placeholder="Türkçe bir yorum yazın...",
        lines=3
    ),
    outputs=gr.Textbox(label="Sentiment"),
    title="Turkish Sentiment Analysis",
    description="Detects positive/negative sentiment in Turkish text. Fine-tuned mBERT on 4,486 Turkish reviews. Part of a cross-lingual transfer experiment — an English-only model scored 41.7% (worse than random) on the same task."
)

demo.launch()